# 📄 Resume Analyser

An AI-powered resume analysis tool using **OpenAI GPT-4o**.

### Features
- 📤 Parse PDF and DOCX resumes
- 🧠 Extract key information (name, email, skills, education, experience)
- 🎯 Match resume against a job description and get a score
- 💡 Suggest missing skills and improvements
- ✅ ATS compatibility check
- 📦 Export analysis as JSON or PDF report

---
**Setup:** Copy `.env.example` to `.env` and add your OpenAI API key.

## 📦 Cell 1 — Install Dependencies

In [ ]:
# Run this cell once to install all required libraries
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'openai', 'PyMuPDF', 'python-docx', 'reportlab', 'python-dotenv', 'ipywidgets'
])
print('✅ All dependencies installed successfully!')

## ⚙️ Cell 2 — Imports & Configuration

In [ ]:
import os
import json
import re
import textwrap
from pathlib import Path
from datetime import datetime

import fitz                          # PyMuPDF — PDF parsing
from docx import Document            # python-docx — DOCX parsing
from openai import OpenAI            # OpenAI GPT API
from dotenv import load_dotenv

from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import cm
from reportlab.lib import colors
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer,
    Table, TableStyle, HRFlowable
)

# ── Load environment variables ──────────────────────────────────────────────
load_dotenv()
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY', '')

if not OPENAI_API_KEY or OPENAI_API_KEY == 'your_openai_api_key_here':
    print('⚠️  WARNING: OPENAI_API_KEY not set!')
    print('   Create a .env file and add: OPENAI_API_KEY=sk-...')
else:
    print('✅ OpenAI API key loaded.')

client = OpenAI(api_key=OPENAI_API_KEY)
EXPORTS_DIR = Path('exports')
EXPORTS_DIR.mkdir(exist_ok=True)
print(f'📁 Exports will be saved to: {EXPORTS_DIR.resolve()}')

## 📂 Cell 3 — Resume File Parser

In [ ]:
def extract_text_from_pdf(file_path: str) -> str:
    """Extract raw text from a PDF file using PyMuPDF."""
    text = ''
    with fitz.open(file_path) as doc:
        for page in doc:
            text += page.get_text()
    return text.strip()


def extract_text_from_docx(file_path: str) -> str:
    """Extract raw text from a DOCX file using python-docx."""
    doc = Document(file_path)
    paragraphs = [p.text for p in doc.paragraphs if p.text.strip()]
    return '\n'.join(paragraphs).strip()


def parse_resume(file_path: str) -> str:
    """Auto-detect file type and extract resume text."""
    path = Path(file_path)
    if not path.exists():
        raise FileNotFoundError(f'File not found: {file_path}')

    ext = path.suffix.lower()
    if ext == '.pdf':
        text = extract_text_from_pdf(file_path)
        print(f'✅ Parsed PDF: {path.name} ({len(text)} chars)')
    elif ext in ('.docx', '.doc'):
        text = extract_text_from_docx(file_path)
        print(f'✅ Parsed DOCX: {path.name} ({len(text)} chars)')
    else:
        raise ValueError(f'Unsupported file type: {ext}. Use PDF or DOCX.')

    if len(text) < 50:
        print('⚠️  Warning: Very little text extracted. The file may be image-based.')
    return text


print('📂 Resume parser functions loaded.')

## 🧠 Cell 4 — AI Analysis Engine (OpenAI GPT)

In [ ]:
# ─────────────────────────────────────────────────────────────
# System prompt instructing GPT to act as a resume analyst
# ─────────────────────────────────────────────────────────────
SYSTEM_PROMPT = """
You are an expert HR consultant and ATS (Applicant Tracking System) specialist
with 15+ years of experience reviewing resumes across all industries.

When given a resume (and optionally a job description), you MUST respond with
a valid JSON object — no markdown, no code fences, just raw JSON.

The JSON must follow this exact schema:
{
  "candidate": {
    "name": "string or null",
    "email": "string or null",
    "phone": "string or null",
    "location": "string or null",
    "linkedin": "string or null",
    "github": "string or null"
  },
  "summary": "2-3 sentence professional summary of the candidate",
  "skills": {
    "technical": ["list of technical skills"],
    "soft": ["list of soft skills"],
    "tools": ["list of tools and software"]
  },
  "education": [
    {
      "degree": "string",
      "field": "string",
      "institution": "string",
      "year": "string or null",
      "gpa": "string or null"
    }
  ],
  "experience": [
    {
      "title": "string",
      "company": "string",
      "duration": "string",
      "highlights": ["key achievements/responsibilities"]
    }
  ],
  "ats_check": {
    "score": 0-100,
    "verdict": "Pass | Fail | Marginal",
    "issues": ["list of ATS issues found"],
    "strengths": ["list of ATS-friendly elements"]
  },
  "job_match": {
    "score": 0-100,
    "verdict": "Excellent | Good | Fair | Poor",
    "matched_keywords": ["keywords found in both resume and JD"],
    "missing_keywords": ["important JD keywords not in resume"]
  },
  "skill_gaps": ["list of skills mentioned in JD but missing in resume"],
  "improvements": [
    {
      "area": "string",
      "suggestion": "actionable suggestion",
      "priority": "High | Medium | Low"
    }
  ],
  "overall_score": 0-100,
  "overall_verdict": "Strong | Average | Needs Work"
}

If no job description is provided, set job_match.score to null, 
job_match.verdict to "N/A", and skill_gaps to [].
Be thorough, honest, and constructive.
""".strip()


def analyse_resume(
    resume_text: str,
    job_description: str = '',
    model: str = 'gpt-4o'
) -> dict:
    """
    Send resume (+ optional JD) to OpenAI GPT and get structured analysis.
    Returns a Python dict parsed from the JSON response.
    """
    user_content = f'RESUME:\n{resume_text}'
    if job_description.strip():
        user_content += f'\n\nJOB DESCRIPTION:\n{job_description}'
    else:
        user_content += '\n\n(No job description provided — skip job match analysis.)'

    print(f'🤖 Sending to {model}... (this may take 15-30 seconds)')
    response = client.chat.completions.create(
        model=model,
        messages=[
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user',   'content': user_content}
        ],
        temperature=0.2,
        response_format={'type': 'json_object'}   # Enforce JSON output
    )

    raw = response.choices[0].message.content
    result = json.loads(raw)
    print(f'✅ Analysis complete! Overall score: {result.get("overall_score", "N/A")}/100')
    return result


print('🧠 AI analysis engine loaded.')

## 🖥️ Cell 5 — Results Display

In [ ]:
def score_badge(score, thresholds=(70, 50)):
    """Return an emoji badge based on score."""
    if score is None or score == 'N/A':
        return '⬜'
    if score >= thresholds[0]:
        return '🟢'
    if score >= thresholds[1]:
        return '🟡'
    return '🔴'


def display_results(analysis: dict):
    """Pretty-print the analysis results in the notebook."""
    sep = '─' * 60

    # ── Header ──────────────────────────────────────────────────
    c = analysis.get('candidate', {})
    name = c.get('name') or 'Unknown Candidate'
    print(f'\n{'═'*60}')
    print(f'  📄 RESUME ANALYSIS REPORT')
    print(f'  👤 {name}')
    print(f'{'═'*60}')

    # ── Contact Info ────────────────────────────────────────────
    print('\n📬 CONTACT INFORMATION')
    print(sep)
    for field in ('email', 'phone', 'location', 'linkedin', 'github'):
        val = c.get(field)
        if val:
            print(f'  {field.capitalize():<10}: {val}')

    # ── Summary ─────────────────────────────────────────────────
    print('\n📝 PROFESSIONAL SUMMARY')
    print(sep)
    summary = analysis.get('summary', 'N/A')
    for line in textwrap.wrap(summary, width=58):
        print(f'  {line}')

    # ── Scores Dashboard ────────────────────────────────────────
    overall = analysis.get('overall_score', 'N/A')
    ats     = analysis.get('ats_check', {}).get('score', 'N/A')
    jm      = analysis.get('job_match', {}).get('score', 'N/A')

    print('\n🏆 SCORES DASHBOARD')
    print(sep)
    print(f'  {score_badge(overall)} Overall Score  : {overall}/100  [{analysis.get("overall_verdict", "")}]')
    print(f'  {score_badge(ats)}    ATS Score      : {ats}/100  [{analysis.get("ats_check", {}).get("verdict", "")}]')
    jm_label = analysis.get('job_match', {}).get('verdict', 'N/A')
    jm_display = f'{jm}/100' if isinstance(jm, int) else 'N/A (no JD provided)'
    print(f'  {score_badge(jm)}    Job Match      : {jm_display}  [{jm_label}]')

    # ── Skills ──────────────────────────────────────────────────
    skills = analysis.get('skills', {})
    print('\n🛠️  SKILLS')
    print(sep)
    for category, items in skills.items():
        if items:
            label = category.replace('_', ' ').title()
            wrapped = textwrap.fill(', '.join(items), width=55, subsequent_indent='              ')
            print(f'  {label:<12}: {wrapped}')

    # ── Education ───────────────────────────────────────────────
    edu = analysis.get('education', [])
    if edu:
        print('\n🎓 EDUCATION')
        print(sep)
        for e in edu:
            gpa = f'  GPA: {e["gpa"]}' if e.get('gpa') else ''
            year = f' ({e["year"]})' if e.get('year') else ''
            print(f'  • {e.get("degree", "")} in {e.get("field", "")}{year}')
            print(f'    {e.get("institution", "")}{gpa}')

    # ── Experience ──────────────────────────────────────────────
    exp = analysis.get('experience', [])
    if exp:
        print('\n💼 EXPERIENCE')
        print(sep)
        for e in exp:
            print(f'  • {e.get("title", "")} @ {e.get("company", "")} [{e.get("duration", "")}]')
            for h in e.get('highlights', [])[:3]:   # Show top 3 highlights
                wrapped = textwrap.fill(h, width=54, subsequent_indent='      ')
                print(f'    – {wrapped}')

    # ── ATS Check ───────────────────────────────────────────────
    ats_data = analysis.get('ats_check', {})
    print('\n✅ ATS COMPATIBILITY CHECK')
    print(sep)
    if ats_data.get('strengths'):
        print('  Strengths:')
        for s in ats_data['strengths']:
            print(f'    ✓ {s}')
    if ats_data.get('issues'):
        print('  Issues:')
        for i in ats_data['issues']:
            print(f'    ✗ {i}')

    # ── Job Match ───────────────────────────────────────────────
    jm_data = analysis.get('job_match', {})
    if jm_data.get('matched_keywords') or jm_data.get('missing_keywords'):
        print('\n🎯 JOB MATCH ANALYSIS')
        print(sep)
        if jm_data.get('matched_keywords'):
            kw = ', '.join(jm_data['matched_keywords'])
            print(f'  ✅ Matched  : {textwrap.fill(kw, 54, subsequent_indent="              ")}')
        if jm_data.get('missing_keywords'):
            kw = ', '.join(jm_data['missing_keywords'])
            print(f'  ❌ Missing  : {textwrap.fill(kw, 54, subsequent_indent="              ")}')

    # ── Skill Gaps ──────────────────────────────────────────────
    gaps = analysis.get('skill_gaps', [])
    if gaps:
        print('\n💡 SKILL GAPS')
        print(sep)
        for g in gaps:
            print(f'  ⚠  {g}')

    # ── Improvements ────────────────────────────────────────────
    improvements = analysis.get('improvements', [])
    if improvements:
        print('\n🚀 IMPROVEMENT SUGGESTIONS')
        print(sep)
        priority_icons = {'High': '🔴', 'Medium': '🟡', 'Low': '🟢'}
        for imp in improvements:
            icon = priority_icons.get(imp.get('priority', ''), '⚪')
            print(f'  {icon} [{imp.get("priority", "")}] {imp.get("area", "")}')
            suggestion = textwrap.fill(imp.get('suggestion', ''), width=55, subsequent_indent='       ')
            print(f'       {suggestion}')

    print(f'\n{'═'*60}\n')


print('🖥️  Display functions loaded.')

## 💾 Cell 6 — Export Functions (JSON & PDF)

In [ ]:
def export_json(analysis: dict, filename: str = None) -> str:
    """Save the analysis result as a JSON file."""
    if not filename:
        ts = datetime.now().strftime('%Y%m%d_%H%M%S')
        name = analysis.get('candidate', {}).get('name', 'resume')
        name = re.sub(r'[^\w]', '_', name or 'resume')
        filename = f'{name}_{ts}.json'

    out_path = EXPORTS_DIR / filename
    with open(out_path, 'w', encoding='utf-8') as f:
        json.dump(analysis, f, indent=2, ensure_ascii=False)
    print(f'💾 JSON saved: {out_path}')
    return str(out_path)


def export_pdf(analysis: dict, filename: str = None) -> str:
    """Generate a professional PDF report from the analysis."""
    if not filename:
        ts = datetime.now().strftime('%Y%m%d_%H%M%S')
        name = analysis.get('candidate', {}).get('name', 'resume')
        name = re.sub(r'[^\w]', '_', name or 'resume')
        filename = f'{name}_{ts}_report.pdf'

    out_path = EXPORTS_DIR / filename

    # ── ReportLab setup ─────────────────────────────────────────
    doc  = SimpleDocTemplate(str(out_path), pagesize=A4,
                             rightMargin=2*cm, leftMargin=2*cm,
                             topMargin=2*cm, bottomMargin=2*cm)
    styles = getSampleStyleSheet()

    # Custom styles
    title_style = ParagraphStyle('Title', parent=styles['Title'],
                                 fontSize=20, textColor=colors.HexColor('#1a73e8'),
                                 spaceAfter=4)
    heading_style = ParagraphStyle('Heading2', parent=styles['Heading2'],
                                   fontSize=13, textColor=colors.HexColor('#1a73e8'),
                                   spaceBefore=10, spaceAfter=4)
    body_style   = ParagraphStyle('Body', parent=styles['Normal'],
                                  fontSize=10, leading=14)
    bullet_style = ParagraphStyle('Bullet', parent=styles['Normal'],
                                  fontSize=10, leading=13, leftIndent=12)

    story = []
    c = analysis.get('candidate', {})
    name = c.get('name', 'Unknown Candidate')

    # ── Title Block ─────────────────────────────────────────────
    story.append(Paragraph('📄 Resume Analysis Report', title_style))
    story.append(Paragraph(f'<b>Candidate:</b> {name}', body_style))
    story.append(Paragraph(f'<b>Generated:</b> {datetime.now().strftime("%B %d, %Y %I:%M %p")}', body_style))
    story.append(HRFlowable(width='100%', thickness=1.5, color=colors.HexColor('#1a73e8')))
    story.append(Spacer(1, 0.3*cm))

    # ── Scores Table ────────────────────────────────────────────
    story.append(Paragraph('Scores', heading_style))
    overall = analysis.get('overall_score', 'N/A')
    ats_sc  = analysis.get('ats_check', {}).get('score', 'N/A')
    jm_sc   = analysis.get('job_match', {}).get('score', 'N/A')
    jm_sc_display = f'{jm_sc}/100' if isinstance(jm_sc, int) else 'N/A'

    score_data = [
        ['Metric', 'Score', 'Verdict'],
        ['Overall Score', f'{overall}/100', analysis.get('overall_verdict', '')],
        ['ATS Compatibility', f'{ats_sc}/100', analysis.get('ats_check', {}).get('verdict', '')],
        ['Job Match', jm_sc_display, analysis.get('job_match', {}).get('verdict', 'N/A')],
    ]
    t = Table(score_data, colWidths=[6*cm, 4*cm, 6*cm])
    t.setStyle(TableStyle([
        ('BACKGROUND',  (0, 0), (-1, 0), colors.HexColor('#1a73e8')),
        ('TEXTCOLOR',   (0, 0), (-1, 0), colors.white),
        ('FONTNAME',    (0, 0), (-1, 0), 'Helvetica-Bold'),
        ('FONTSIZE',    (0, 0), (-1, 0), 10),
        ('ROWBACKGROUNDS', (0, 1), (-1, -1), [colors.HexColor('#f8f9fa'), colors.white]),
        ('GRID',        (0, 0), (-1, -1), 0.5, colors.HexColor('#dee2e6')),
        ('ALIGN',       (0, 0), (-1, -1), 'CENTER'),
        ('VALIGN',      (0, 0), (-1, -1), 'MIDDLE'),
        ('TOPPADDING',  (0, 0), (-1, -1), 6),
        ('BOTTOMPADDING',(0, 0), (-1, -1), 6),
    ]))
    story.append(t)
    story.append(Spacer(1, 0.4*cm))

    # ── Helper to add section ────────────────────────────────────
    def add_section(title, items, icon='•'):
        if not items:
            return
        story.append(Paragraph(title, heading_style))
        for item in items:
            story.append(Paragraph(f'{icon} {item}', bullet_style))
        story.append(Spacer(1, 0.2*cm))

    # ── Summary ─────────────────────────────────────────────────
    summary = analysis.get('summary', '')
    if summary:
        story.append(Paragraph('Professional Summary', heading_style))
        story.append(Paragraph(summary, body_style))
        story.append(Spacer(1, 0.3*cm))

    # ── Contact Info ────────────────────────────────────────────
    story.append(Paragraph('Contact Information', heading_style))
    for field in ('email', 'phone', 'location', 'linkedin', 'github'):
        val = c.get(field)
        if val:
            story.append(Paragraph(f'<b>{field.capitalize()}:</b> {val}', body_style))
    story.append(Spacer(1, 0.3*cm))

    # ── Skills ──────────────────────────────────────────────────
    skills = analysis.get('skills', {})
    story.append(Paragraph('Skills', heading_style))
    for cat, items in skills.items():
        if items:
            label = cat.replace('_', ' ').title()
            story.append(Paragraph(f'<b>{label}:</b> {', '.join(items)}', body_style))
    story.append(Spacer(1, 0.3*cm))

    # ── Education ───────────────────────────────────────────────
    edu = analysis.get('education', [])
    if edu:
        story.append(Paragraph('Education', heading_style))
        for e in edu:
            gpa = f'  |  GPA: {e["gpa"]}' if e.get('gpa') else ''
            year = f' ({e["year"]})' if e.get('year') else ''
            story.append(Paragraph(
                f'<b>{e.get("degree", "")} in {e.get("field", "")}</b>{year}{gpa}', body_style))
            story.append(Paragraph(e.get('institution', ''), bullet_style))
        story.append(Spacer(1, 0.3*cm))

    # ── Experience ──────────────────────────────────────────────
    exp = analysis.get('experience', [])
    if exp:
        story.append(Paragraph('Work Experience', heading_style))
        for e in exp:
            story.append(Paragraph(
                f'<b>{e.get("title", "")} @ {e.get("company", "")}</b> [{e.get("duration", "")}]', body_style))
            for h in e.get('highlights', []):
                story.append(Paragraph(f'• {h}', bullet_style))
        story.append(Spacer(1, 0.3*cm))

    # ── ATS Issues & Strengths ──────────────────────────────────
    ats_data = analysis.get('ats_check', {})
    add_section('ATS Strengths', ats_data.get('strengths', []), '✓')
    add_section('ATS Issues', ats_data.get('issues', []), '✗')

    # ── Job Match ───────────────────────────────────────────────
    jm_data = analysis.get('job_match', {})
    if jm_data.get('matched_keywords'):
        add_section('Matched Keywords', jm_data['matched_keywords'], '✅')
    if jm_data.get('missing_keywords'):
        add_section('Missing Keywords', jm_data['missing_keywords'], '❌')

    # ── Skill Gaps ──────────────────────────────────────────────
    add_section('Skill Gaps', analysis.get('skill_gaps', []), '⚠')

    # ── Improvement Suggestions ─────────────────────────────────
    improvements = analysis.get('improvements', [])
    if improvements:
        story.append(Paragraph('Improvement Suggestions', heading_style))
        priority_map = {'High': '🔴', 'Medium': '🟡', 'Low': '🟢'}
        for imp in improvements:
            icon = priority_map.get(imp.get('priority', ''), '•')
            story.append(Paragraph(
                f'<b>{icon} [{imp.get("priority","")}] {imp.get("area","")}</b>: {imp.get("suggestion","")}',
                bullet_style))

    # ── Build PDF ───────────────────────────────────────────────
    doc.build(story)
    print(f'📄 PDF saved: {out_path}')
    return str(out_path)


print('💾 Export functions loaded.')

## 🚀 Cell 7 — Run the Analysis

**Instructions:**
1. Set `RESUME_FILE` to the path of your PDF or DOCX resume
2. Optionally paste a job description into `JOB_DESCRIPTION`
3. Run this cell

In [ ]:
# ─────────────────────────────────────────────────────────────
#  ✏️  CONFIGURE YOUR INPUTS HERE
# ─────────────────────────────────────────────────────────────

RESUME_FILE = 'sample_resume.pdf'   # ← Path to your resume (PDF or DOCX)

JOB_DESCRIPTION = """
We are looking for a Senior Software Engineer with strong experience in:
- Python and backend development (FastAPI, Django)
- Cloud platforms (AWS, GCP)
- Machine Learning and data pipelines
- Docker, Kubernetes, CI/CD pipelines
- Strong communication and team leadership skills
- 5+ years of experience in software development
""".strip()

# Leave JOB_DESCRIPTION empty string to skip job match analysis:
# JOB_DESCRIPTION = ''

# ─────────────────────────────────────────────────────────────
#  🚀 Run Analysis
# ─────────────────────────────────────────────────────────────
resume_text = parse_resume(RESUME_FILE)
analysis    = analyse_resume(resume_text, JOB_DESCRIPTION)
display_results(analysis)

## 💾 Cell 8 — Export Results

In [ ]:
# Export analysis as JSON
json_path = export_json(analysis)

# Export analysis as a PDF report
pdf_path  = export_pdf(analysis)

print(f'\n✅ Exports ready:')
print(f'   JSON : {json_path}')
print(f'   PDF  : {pdf_path}')

## 🔄 Cell 9 — Batch Analysis (Multiple Resumes)

In [ ]:
def batch_analyse(resume_files: list, job_description: str = '') -> list:
    """
    Analyse multiple resumes and export individual reports.
    Returns a summary table of all candidates with their scores.
    """
    results = []
    total = len(resume_files)

    for i, file_path in enumerate(resume_files, 1):
        print(f'\n[{i}/{total}] Processing: {file_path}')
        print('─' * 50)
        try:
            text     = parse_resume(file_path)
            analysis = analyse_resume(text, job_description)
            export_json(analysis)
            export_pdf(analysis)
            results.append({
                'file':            Path(file_path).name,
                'name':            analysis.get('candidate', {}).get('name', 'Unknown'),
                'email':           analysis.get('candidate', {}).get('email', 'N/A'),
                'overall_score':   analysis.get('overall_score', 'N/A'),
                'ats_score':       analysis.get('ats_check', {}).get('score', 'N/A'),
                'job_match_score': analysis.get('job_match', {}).get('score', 'N/A'),
                'verdict':         analysis.get('overall_verdict', 'N/A'),
                'analysis':        analysis
            })
        except Exception as e:
            print(f'  ❌ Error processing {file_path}: {e}')
            results.append({'file': Path(file_path).name, 'error': str(e)})

    # ── Print Summary Table ──────────────────────────────────────
    print(f'\n{'═'*80}')
    print('  BATCH ANALYSIS SUMMARY')
    print(f'{'═'*80}')
    header = f'  {"File":<25} {"Name":<20} {"Overall":>7} {"ATS":>5} {"JobMatch":>9} {"Verdict":<12}'
    print(header)
    print('─' * 80)
    for r in results:
        if 'error' in r:
            print(f'  {r["file"]:<25} ERROR: {r["error"]}')
        else:
            jm = r.get('job_match_score', 'N/A')
            jm_str = f'{jm}/100' if isinstance(jm, int) else 'N/A'
            print(f'  {r["file"]:<25} {r["name"]:<20} {str(r["overall_score"]) + "/100":>7}',
                  f'{str(r["ats_score"]) + "/100":>8} {jm_str:>9} {r["verdict"]:<12}')
    print(f'{'═'*80}')

    # Sort by overall score (descending)
    sorted_results = sorted(
        [r for r in results if 'error' not in r],
        key=lambda x: x.get('overall_score', 0),
        reverse=True
    )
    if sorted_results:
        top = sorted_results[0]
        print(f'\n🏆 Top Candidate: {top["name"]} ({top["overall_score"]}/100)')

    return results


# ── Example usage (uncomment and edit paths to run) ──────────
# batch_results = batch_analyse(
#     resume_files=['resume1.pdf', 'resume2.docx', 'resume3.pdf'],
#     job_description=JOB_DESCRIPTION
# )

print('🔄 Batch analysis function loaded.')
print('  → Uncomment the example at the bottom and provide your file paths to run batch analysis.')